# DataFrame Regression Baseline (FlashSequential)

Generated by `flashkeras.notebooks`.

This baseline expects a CSV with numeric feature columns and one numeric target column.
Edit the **Parameters** cell below, then `Run All`.

In [ ]:
data_csv = "REPLACE_ME/path/to/data.csv"
target_column = "target"
separator = ","
test_size = 0.2
epochs = 50
batch_size = 32
seed = 42

## 1. Load and split data

In [ ]:
import pandas as pd
from flashkeras.data_collecting import read_csv
from flashkeras.preprocessing import FlashPreProcessing
from flashkeras.preprocessing.tabular.features import minMaxScaler

dataframe = read_csv(data_csv, sep=separator)
if target_column not in dataframe.columns:
    raise KeyError(f"Target column not found: {target_column}")

features = dataframe.drop(columns=[target_column])
if not all(pd.api.types.is_numeric_dtype(column) for column in features.dtypes):
    raise TypeError("All feature columns must be numeric for this baseline.")
x_scaler, features = minMaxScaler(features, return_scaler=True)

target = dataframe[target_column]
if not pd.api.types.is_numeric_dtype(target):
    raise TypeError("The target column must be numeric for regression.")
y_scaler, target = minMaxScaler(target, return_scaler=True)

x_train, x_test, y_train, y_test = FlashPreProcessing.train_test_split(
    features, target, test_split=test_size, random_state=seed
)

print(f"Training samples: {len(y_train)}")
print(f"Test samples: {len(y_test)}")
print(f"Features: {x_train.shape[1]}")

In [ ]:
x_train

In [ ]:
y_train

## 2. Build baseline regressor

In [ ]:
from flashkeras.models import FlashSequential

flash = FlashSequential("regression")
flash.addDense(64, activation="relu")
flash.addDense(32, activation="relu")

## 3. Train

In [ ]:
history = flash.train(
    x=x_train,
    y=y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(x_test, y_test),
    auto_output_layer=True
)

## 4. Training curves

In [ ]:
from flashkeras.analysing.graphs.evaluation import plot_history_train_curve

plot_history_train_curve(history)

## 5. Evaluate

In [ ]:
from flashkeras.evaluation import getMAE, getMSE, getRMSE

mae = getMAE(flash, x_test, y_test)
mse = getMSE(flash, x_test, y_test)
rmse = getRMSE(flash, x_test, y_test)

print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")

## Next steps

- Compare against a linear regression baseline
- Tune the hidden layers, learning rate, and batch size
- Add categorical encoding if the dataset contains non-numeric features
- Save the model with `flash.model.save("model.keras")`